# Architecture localization — frozen visual readouts

Compare **A: eight pooled scores**, **B: spatial score maps**, and **C: visual tokens** using the same frozen `AttributeBindingViT` checkpoints. Train new readouts on the original Medium World training questions, then evaluate identical ID-development questions and controlled image edits. No pretrained weights or optimizer states are updated.

The purpose is to locate where useful Shape and Color–Shape association evidence becomes difficult to use. An improved readout is *diagnostic evidence*, not proof of an isolated mechanism: the input information and inductive bias differ across A/B/C.

In [1]:
from pathlib import Path
import sys, json, hashlib, math
from collections import Counter, defaultdict

PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path: 
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from IPython.display import display
from src.medium_visual_world.learning import MediumDataset, AttributeBindingViT, encode_query
from src.medium_visual_world.diagnostic_followup import load_frozen_model, count_free, make_swap
from src.medium_visual_world.representation_probing import counterfactual_rows, oracle, query_tokens
from src.medium_visual_world.world import SHAPES, from_dict
from src.medium_visual_world.render import render
from src.medium_visual_world.generate import HOLDOUT

SCHEDULES = ("mixed", "prereq_mixed_pair_focus")
SEEDS = (42, 43, 44, 45, 46)
STEPS = (5400, 7200, 9000)
READOUTS = ("scalar", "spatial", "tokens")
DEVICE = "cuda"
EPOCHS, BATCH_SIZE, LR = 30, 128, 1e-3
FEATURE_BATCH_SIZE, EVAL_BATCH_SIZE = 48, 128
READOUT_SEED_OFFSET = 20261113
CF_SCENE_LIMIT, CF_SELECTION_SEED = 128, 19
DATA_DIR = PROJECT_ROOT / "outputs/medium_world/dataset"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs/medium_world/experiments/representation_diagnostics/five_seeds/checkpoints"
OUT_DIR = PROJECT_ROOT / "outputs/medium_world/experiments/architecture_localization"

train_data = MediumDataset(DATA_DIR / "train.jsonl")
id_dev = MediumDataset(DATA_DIR / "id_dev.jsonl")
print("Training questions:", len(train_data), "| ID-dev questions:", len(id_dev), "| Device:",DEVICE)

Training questions: 10000 | ID-dev questions: 1024 | Device: cuda


## 1. Three readouts

- **Scalar:** new classifier on the original eight max/mean summaries.
- **Spatial:** 3×3 convolutions on the original four 16×16 query-score maps before pooling.
- **Tokens:** new query-conditioned spatial readout on the frozen 256×48 visual tokens.

All three are trained separately for 30 epochs with the same question order, optimizer and learning rate. Their parameter counts are similar, but they are *not* functionally or computationally equivalent. The original pretrained classifier is retained as a reference (without extra training).

In [2]:
class ScalarReadout(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(8, 256), nn.GELU(), nn.Linear(256, 2))

    def forward(self, eight, maps, tokens, query): 
        return self.net(eight)

class SpatialReadout(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.head = nn.Linear(32, 2)

    def forward(self, eight, maps, tokens, query):
        z = F.gelu(self.conv1(maps))
        z = F.gelu(self.conv2(z))
        return self.head(torch.cat((z.amax((2, 3)), z.mean((2, 3))), dim=1))

class TokenReadout(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual = nn.Linear(48, 10)
        self.condition = nn.Linear(144, 10)
        self.conv = nn.Conv2d(10, 10, kernel_size=3, padding=1)
        self.head = nn.Linear(20, 2)

    def forward(self, eight, maps, tokens, query):
        z = F.gelu(self.visual(tokens) + self.condition(query)[:, None, :])
        z = F.gelu(self.conv(z.transpose(1, 2).reshape(-1, 10, 16, 16)))
        return self.head(torch.cat((z.amax((2, 3)), z.mean((2, 3))), dim=1))

READOUT_CLASSES = {"scalar": ScalarReadout, "spatial": SpatialReadout, "tokens": TokenReadout}
parameters = {name:sum(p.numel() for p in cls().parameters()) for name,cls in READOUT_CLASSES.items()}
display(pd.Series(parameters,name="trainable_parameters").to_frame())

,trainable_parameters
scalar,2818
spatial,2978
tokens,2892


## 2. Frozen feature extraction

Extract each scene's visual tokens once per checkpoint and cache them on CPU in float32. For each question, compute the *same* attribute scores, joint-query scores and local interaction as the original network. Preserve both the complete maps and their original eight scalar reductions. This avoids repeating ViT inference when the same image appears in multiple questions.

In [3]:
@torch.no_grad()
def question_features(model, tokens, q):
    e = model.emb(q[:, 1:4]); joint = model.query(e.flatten(1))
    scale = math.sqrt(tokens.shape[-1])
    s1 = torch.einsum("bnd,bd->bn", tokens, e[:, 1]) / scale
    s2 = torch.einsum("bnd,bd->bn", tokens, e[:, 2]) / scale
    sj = torch.einsum("bnd,bd->bn", tokens, joint) / scale
    interaction = torch.tanh(s1) * torch.tanh(s2)
    maps = torch.stack((s1, s2, sj, interaction), dim=1).reshape(-1, 4, 16, 16)
    eight = torch.stack((s1.amax(1), s1.mean(1), s2.amax(1), s2.mean(1),
                         sj.amax(1), sj.mean(1), interaction.amax(1), interaction.mean(1)), dim=-1)
    return eight, maps, e.flatten(1)

@torch.no_grad()
def cache_features(model, dataset, device):
    scenes = []
    for start in range(0, len(dataset.images), FEATURE_BATCH_SIZE):
        x = dataset.images[start:start+FEATURE_BATCH_SIZE].to(device=device,dtype=torch.float32).div(255)
        p = model.patch(x).flatten(2).transpose(1, 2)
        v = model.score_norm(model.pre_norm(p) + .5 * model.post_norm(model.encoder(p + model.pos)))
        scenes.append(v.float().cpu())
    tokens = torch.cat(scenes)
    eight_parts, map_parts, query_parts = [], [], []
    for start in range(0, len(dataset), FEATURE_BATCH_SIZE):
        ids = dataset.image_indices[start:start+FEATURE_BATCH_SIZE]
        v = tokens[ids].to(device=device,dtype=torch.float32)
        q = dataset.queries[start:start+FEATURE_BATCH_SIZE].to(device)
        eight, maps, query = question_features(model, v, q)
        eight_parts.append(eight.cpu()); map_parts.append(maps.cpu()); query_parts.append(query.cpu())
    return {"tokens":tokens, "image_index":dataset.image_indices,
            "eight":torch.cat(eight_parts), "maps":torch.cat(map_parts),
            "query":torch.cat(query_parts), "labels":dataset.labels}

def readout_batch(cache, indices, name, device):
    if name == "scalar": return cache["eight"][indices].to(device), None, None, None
    if name == "spatial": return None, cache["maps"][indices].to(device), None, None
    scenes = cache["image_index"][indices]
    return None, None, cache["tokens"][scenes].to(device), cache["query"][indices].to(device)

## 3. Fit and evaluate the replacement readouts

Each frozen checkpoint gets three independently fitted readers. Epochs, minibatch permutations and optimizer settings are fixed across A/B/C. No validation-based checkpoint selection or gradient updates to the original model.

In [4]:
def fit_readout(cache, name, seed, device):
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    net = READOUT_CLASSES[name]().to(device)
    optimizer = torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=0)
    n = len(cache["labels"])
    for epoch in range(EPOCHS):
        order = torch.randperm(n,generator=torch.Generator().manual_seed(seed+epoch))
        net.train()
        for batch in order.split(BATCH_SIZE):
            args = readout_batch(cache,batch,name,device)
            y = cache["labels"][batch].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(net(*args),y)
            loss.backward(); optimizer.step()
    return net.eval()

@torch.no_grad()
def predict_readouts(model,heads,cache,device,original=True):
    names = ("pretrained",*heads) if original else tuple(heads)
    predictions = {k:[] for k in names}
    for start in range(0,len(cache["labels"]),EVAL_BATCH_SIZE):
        ids = torch.arange(start,min(start+EVAL_BATCH_SIZE,len(cache["labels"])))
        if original:
            eight = cache["eight"][ids].to(device)
            predictions["pretrained"].append(model.head(eight).softmax(-1)[:,1].cpu().numpy())
        for name,head in heads.items():
            args = readout_batch(cache,ids,name,device)
            predictions[name].append(head(*args).softmax(-1)[:,1].cpu().numpy())
    return {k:np.concatenate(v) for k,v in predictions.items()}

## 4. Counterfactual scenes

Use the existing validated intervention builders. **Strict Binding swaps** preserve the exact Color and Shape counts and flip the Binding label. Additional edits test whether Shape and Conjunction respond correctly when an attribute is inserted, and whether Shape answers remain correct when only Color changes. Recoloring uses both present and absent Shape queries so an always-Yes or always-No predictor cannot pass the combined test merely by staying invariant.

In [5]:
def make_controlled_edits(dev_rows):
    edits=[]
    for row in dev_rows:
        if not count_free(row): continue
        pair=make_swap(row)
        if pair is None: continue
        key="strict|"+str(row["id"])
        for state,scene,label in (("before",pair["before"],pair["label_before"]),
                                   ("after",pair["after"],pair["label_after"])):
            edits.append(dict(pair_id=key,scene_id=row["scene_id"],scene_key=key+"|"+state,
                              edit="binding_swap",task="pair",color=pair["target_color"],
                              shape=pair["target_shape"],state=state,scene=scene,label=label,group="strict_swap"))

    generated=counterfactual_rows(dev_rows,limit=CF_SCENE_LIMIT,seed=CF_SELECTION_SEED,split="id_dev")
    by_pair=defaultdict(dict)
    for row in generated: by_pair[row["pair_id"]][row["state"]]=row
    for source_id,pair in sorted(by_pair.items()):
        before,after=pair["before"],pair["after"]
        edit=before["edit"]
        tasks={"binding_swap":("pair","color","shape","conjunction"),
               "shape_insert":("shape","conjunction","color"),
               "color_insert":("color","conjunction","shape")}[edit]
        for task in tasks:
            key="grid|"+source_id+"|"+task
            for state,source in (("before",before),("after",after)):
                label=oracle(source["scene"],task,source["color"],source["shape"])
                edits.append(dict(pair_id=key,scene_id=source["scene_id"],scene_key="grid|"+source_id+"|"+state,
                                  edit=edit,task=task,color=source["color"],shape=source["shape"],
                                  state=state,scene=source["scene"],label=label,group="grid"))
        if edit=="color_insert":
            for shape in SHAPES:
                key="shape_recolor|"+source_id+"|"+shape
                for state,source in (("before",before),("after",after)):
                    edits.append(dict(pair_id=key,scene_id=source["scene_id"],
                                      scene_key="grid|"+source_id+"|"+state,
                                      edit=edit,task="shape",color=source["color"],shape=shape,
                                      state=state,scene=source["scene"],
                                      label=oracle(source["scene"],"shape",source["color"],shape),
                                      group="shape_recolor_all"))

    heldout_flags={}
    for key,group in pd.DataFrame(edits).groupby("pair_id"):
        a,b=group.set_index("state").loc[["before","after"]].to_dict("records")
        if a["edit"]=="binding_swap":
            assert Counter(o["color"] for o in a["scene"]["objects"])==Counter(o["color"] for o in b["scene"]["objects"])
            assert Counter(o["shape"] for o in a["scene"]["objects"])==Counter(o["shape"] for o in b["scene"]["objects"])
        before_heldout=any((o["color"],o["shape"]) in HOLDOUT for o in a["scene"]["objects"])
        after_heldout=any((o["color"],o["shape"]) in HOLDOUT for o in b["scene"]["objects"])
        heldout_flags[key]=int(after_heldout and not before_heldout)
    for row in edits:row["heldout_created"]=heldout_flags[row["pair_id"]]
    return edits

class EditedDataset:
    """Same fields as MediumDataset; each rendered edited scene is cached once."""
    def __init__(self,rows):
        self.rows=rows;scene_lookup={};images=[];indices=[];queries=[];labels=[]
        for row in rows:
            key=row["scene_key"]
            if key not in scene_lookup:
                pixels=np.array(render(from_dict(row["scene"])).convert("RGB"),dtype=np.uint8)
                scene_lookup[key]=len(images);images.append(pixels.transpose(2,0,1).copy())
            indices.append(scene_lookup[key])
            queries.append(encode_query(query_tokens(row["task"],row["color"],row["shape"])))
            labels.append(int(row["label"]))
        self.images=torch.from_numpy(np.stack(images))
        self.image_indices=torch.tensor(indices,dtype=torch.long)
        self.queries=torch.tensor(queries,dtype=torch.long)
        self.labels=torch.tensor(labels,dtype=torch.long)

    def __len__(self):return len(self.rows)

edits=make_controlled_edits(id_dev.rows)
edit_data=EditedDataset(edits)
print("Controlled endpoint questions:",len(edits),"| Unique rendered scenes:",len(edit_data.images))

Controlled endpoint questions: 16622 | Unique rendered scenes: 3418


## 5. Measurements

Report ordinary task accuracy, the **52 ID-dev Conjunction negatives with Color present / Shape absent**, count-controlled Binding balanced accuracy, strict Binding-swap both-correct and direction scores, shape-insertion behavior, and recoloring invariance. Evaluate all readouts on identical example IDs. For controlled pairs, keep both endpoint probabilities in the saved records.

In [6]:
def binary_summary(labels,prob):
    y=np.asarray(labels,dtype=int);p=np.asarray(prob,dtype=float)
    pred=(p>=.5).astype(int);yes=y==1;no=y==0
    tpr=float((pred[yes]==1).mean()) if yes.any() else np.nan
    tnr=float((pred[no]==0).mean()) if no.any() else np.nan
    return dict(n=len(y),n_yes=int(yes.sum()),n_no=int(no.sum()),accuracy=float((pred==y).mean()),
                tpr=tpr,tnr=tnr,balanced_accuracy=(tpr+tnr)/2,
                pred_yes=float(pred.mean()))

def id_results(dataset,probs):
    frame=pd.DataFrame({"example_id":[str(r["id"]) for r in dataset.rows],
                        "scene_id":[r["scene_id"] for r in dataset.rows],
                        "task":[r["template"] for r in dataset.rows],
                        "label":dataset.labels.numpy()})
    for name,p in probs.items():frame["p_"+name]=p
    controlled=np.array([count_free(r) for r in dataset.rows])
    missing=np.array([r["template"]=="conjunction" and not r["label"] and
                      any(o["color"]==r["question"]["args"][0] for o in r["scene"]["objects"]) and
                      not any(o["shape"]==r["question"]["args"][1] for o in r["scene"]["objects"])
                      for r in dataset.rows])
    selectors={**{"id_"+t:frame.task.eq(t).to_numpy() for t in ("color","shape","conjunction","pair")},
               "id_pair_count_controlled":controlled,"id_conjunction_missing_shape":missing}
    summary=[]
    for name in probs:
        for metric,mask in selectors.items():
            summary.append(dict(readout=name,metric=metric,
                                **binary_summary(frame.loc[mask,"label"],frame.loc[mask,"p_"+name])))
    return pd.DataFrame(summary),frame

def edit_results(dataset,probs):
    frame=pd.DataFrame([{k:v for k,v in r.items() if k!="scene"} for r in dataset.rows])
    for name,p in probs.items():frame["p_"+name]=p
    pairs=[]
    for pair_id,part in frame.groupby("pair_id"):
        a,b=part.set_index("state").loc[["before","after"]].to_dict("records")
        y0,y1=int(a["label"]),int(b["label"]);flips=y0!=y1
        for name in probs:
            p0,p1=float(a["p_"+name]),float(b["p_"+name])
            pairs.append(dict(pair_id=pair_id,scene_id=a["scene_id"],readout=name,group=a["group"],
                              edit=a["edit"],task=a["task"],heldout_created=a["heldout_created"],
                              original_label=y0,label_flips=int(flips),
                              both_correct=int(((p0>=.5)==bool(y0)) and ((p1>=.5)==bool(y1))),
                              correct_direction=int((p1-p0)*(y1-y0)>0) if flips else np.nan,
                              invariant_pred=int((p0>=.5)==(p1>=.5)) if not flips else np.nan,
                              abs_probability_change=abs(p1-p0)))
    paired=pd.DataFrame(pairs)
    masks={"strict_swap_all":paired.group.eq("strict_swap"),
           "strict_swap_id_safe":paired.group.eq("strict_swap") & paired.heldout_created.eq(0)}
    for edit in ("binding_swap","shape_insert","color_insert"):
        for task in ("pair","shape","conjunction","color"):
            mask=paired.group.eq("grid") & paired.edit.eq(edit) & paired.task.eq(task) & paired.heldout_created.eq(0)
            if mask.any():masks["grid_"+edit+"_"+task+"_id_safe"]=mask
    recolor=paired.group.eq("shape_recolor_all") & paired.heldout_created.eq(0)
    masks.update({"shape_recolor_all_id_safe":recolor,
                  "shape_recolor_absent_id_safe":recolor & paired.original_label.eq(0),
                  "shape_recolor_present_id_safe":recolor & paired.original_label.eq(1)})
    summary=[]
    for metric,mask in masks.items():
        for name in probs:
            part=paired[mask & paired.readout.eq(name)]
            if part.empty:continue
            flips=part[part.label_flips.eq(1)];invariants=part[part.label_flips.eq(0)]
            summary.append(dict(readout=name,metric=metric,n_pairs=len(part),
                                both_correct=float(part.both_correct.mean()),
                                correct_direction=float(flips.correct_direction.mean()) if len(flips) else np.nan,
                                invariant_prediction=float(invariants.invariant_pred.mean()) if len(invariants) else np.nan,
                                mean_abs_probability_change=float(part.abs_probability_change.mean())))
    return pd.DataFrame(summary),paired,frame

## 6. Run all 30 frozen-checkpoint experiments

Each execution fits three new readers at one original checkpoint and saves per-question ID and edit predictions, aggregate metrics, and the new reader weights. The historical ViT checkpoint remains untouched.

In [7]:
def fit_and_save(model,folder,provenance,original=True):
    device=torch.device(DEVICE)
    train_cache=cache_features(model,train_data,device)
    dev_cache=cache_features(model,id_dev,device)
    edit_cache=cache_features(model,edit_data,device)
    heads={}
    for name in READOUTS:
        print("   Fitting",name,flush=True)
        heads[name]=fit_readout(train_cache,name,READOUT_SEED_OFFSET+provenance["seed"],device)
    id_prob=predict_readouts(model,heads,dev_cache,device,original)
    edit_prob=predict_readouts(model,heads,edit_cache,device,original)
    id_metrics,id_predictions=id_results(id_dev,id_prob)
    edit_metrics,edit_pairs,edit_predictions=edit_results(edit_data,edit_prob)
    folder.mkdir(parents=True,exist_ok=True)
    for name,frame in (("id_metrics",id_metrics),("id_predictions",id_predictions),
                       ("edit_metrics",edit_metrics),("edit_pairs",edit_pairs),
                       ("edit_predictions",edit_predictions)):
        frame.to_csv(folder/(name+".csv"),index=False)
    weights=folder/"readouts";weights.mkdir(exist_ok=True)
    for name,net in heads.items():
        torch.save({"kind":name,"state_dict":{k:v.cpu() for k,v in net.state_dict().items()},
                    "parameters":parameters[name]},weights/(name+".pt"))
    (folder/"run.json").write_text(json.dumps({**provenance,"epochs":EPOCHS,"batch_size":BATCH_SIZE,
                       "lr":LR,"readout_params":parameters},indent=2)+"\n",encoding="utf-8")
    return {**provenance,"output":str(folder)}

runs=[]
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            path=CHECKPOINT_DIR/f"{schedule}_constant_seed_{seed}_step_{step}.pt"
            print("Checkpoint:",schedule,seed,step,flush=True)
            model,_=load_frozen_model(path,DEVICE)
            record=dict(schedule=schedule,seed=seed,step=step,
                        source_checkpoint_sha256=hashlib.sha256(path.read_bytes()).hexdigest())
            folder=OUT_DIR/"frozen_heads"/f"{schedule}_seed{seed}_step{step}"
            runs.append(fit_and_save(model,folder,record))
            del model

display(pd.DataFrame(runs))

Checkpoint: mixed 42 5400
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 42 7200
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 42 9000
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 43 5400
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 43 7200
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 43 9000
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 44 5400
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 44 7200
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 44 9000
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 45 5400
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 45 7200
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 45 9000
   Fitting scalar
   Fitting spatial
   Fitting tokens
Checkpoint: mixed 46 5400
  

,schedule,seed,step,source_checkpoint_sha256,output
0,mixed,42,5400,1a5f298404afbfe33c736a02a8b49b4fb77aab7bcb62ce...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
1,mixed,42,7200,eb3e098b5ac9ef520d93d7359a4b74b79b806de5745fcd...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
2,mixed,42,9000,ce6a40dcda1c502ace8a3bfaec368b79772e00796e01f6...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
3,mixed,43,5400,c926ae40f795bbe0c54a3fa3bb60f52f975e4dad418728...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
4,mixed,43,7200,8d0dcc9706e4abbf5b01a93d5de53a914f89c4841f3e3e...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
5,mixed,43,9000,a3c7e979fcef434e6b949f7215b64b0d1ea02f394a552e...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
6,mixed,44,5400,80339ce336bf688d27020d049fcde4698d0fa367033605...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
7,mixed,44,7200,ca2063a0616c880de7ac3176c0dd9a8aed1f405afc5bee...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
8,mixed,44,9000,7387a9f1a497e2eeb5e669ea0aeb5fe57b7fe97896eb43...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
9,mixed,45,5400,4251f7d5185c097270a625ba673764bcaa32c901a91845...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...


## 7. Random-visual controls

For each seed, use the trained Mixed step-9,000 query embeddings/projector but replace the entire visual pathway with a freshly initialized, **frozen** `AttributeBindingViT` visual encoder. Train the same three new readers. A strong result with this control indicates how much the decoder can recover from preserved low-level visual information without visual-encoder learning. The original classifier is excluded for this condition.

In [8]:
random_runs=[]
for seed in SEEDS:
    print("Random visual:",seed,flush=True)
    source=CHECKPOINT_DIR/f"mixed_constant_seed_{seed}_step_9000.pt"
    trained,_=load_frozen_model(source,DEVICE)
    torch.manual_seed(300000+seed)
    random_model=AttributeBindingViT(dim=48,patch_size=4,depth=1)
    random_model.emb.load_state_dict(trained.emb.state_dict())
    random_model.query.load_state_dict(trained.query.state_dict())
    random_model.requires_grad_(False)
    random_model=random_model.to(DEVICE).eval()
    folder=OUT_DIR/"random_visual"/f"random_visual_seed{seed}"
    record=dict(seed=seed,step=9000,control="random_visual_trained_query",
                source_checkpoint_sha256=hashlib.sha256(source.read_bytes()).hexdigest())
    random_runs.append(fit_and_save(random_model,folder,record,original=False))
    del trained,random_model

display(pd.DataFrame(random_runs))

Random visual: 42
   Fitting scalar
   Fitting spatial
   Fitting tokens
Random visual: 43
   Fitting scalar
   Fitting spatial
   Fitting tokens
Random visual: 44
   Fitting scalar
   Fitting spatial
   Fitting tokens
Random visual: 45
   Fitting scalar
   Fitting spatial
   Fitting tokens
Random visual: 46
   Fitting scalar
   Fitting spatial
   Fitting tokens


,seed,step,control,source_checkpoint_sha256,output
0,42,9000,random_visual_trained_query,ce6a40dcda1c502ace8a3bfaec368b79772e00796e01f6...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
1,43,9000,random_visual_trained_query,a3c7e979fcef434e6b949f7215b64b0d1ea02f394a552e...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
2,44,9000,random_visual_trained_query,7387a9f1a497e2eeb5e669ea0aeb5fe57b7fe97896eb43...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
3,45,9000,random_visual_trained_query,ff9cdd89b12fa65bfe91201fde7dcc3f053fa17e41c84f...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
4,46,9000,random_visual_trained_query,4dafb7fe2e9ce30c5829095fa901ccec89a89c44cf8a82...,c:\Users\Quoc Thai\Downloads\AI\AIO\Conquer\AI...
